In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

In [2]:
def explorer_dataset(X, y, mask=None):
    if mask is not None:
        X = X[mask]
        y = y[mask]

    print("Forme des données :", X.shape)
    print("Classes présentes :", set(y))

    for classe in set(y):
        nombre = sum(y == classe)
        pourcentage = nombre / len(y) * 100
        print(f"Classe {classe} : {nombre} cas ({pourcentage:.2f} %)")

In [3]:
X, y = load_breast_cancer(return_X_y=True)

## Test 1

In [4]:
explorer_dataset(X, y)

Forme des données : (569, 30)
Classes présentes : {np.int64(0), np.int64(1)}
Classe 0 : 212 cas (37.26 %)
Classe 1 : 357 cas (62.74 %)


## Test 2

In [5]:
mask = y == 0
explorer_dataset(X, y, mask)

Forme des données : (212, 30)
Classes présentes : {np.int64(0)}
Classe 0 : 212 cas (100.00 %)


## Test 3 

In [6]:
indices = np.concatenate([
    np.where(y == 0)[0][:5],
    np.where(y == 1)[0][:95]
])

explorer_dataset(X[indices], y[indices])

Forme des données : (100, 30)
Classes présentes : {np.int64(0), np.int64(1)}
Classe 0 : 5 cas (5.00 %)
Classe 1 : 95 cas (95.00 %)


# Phase 2 

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    predictions = modele.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    return accuracy

In [9]:
modele = LogisticRegression(max_iter=10000)
resultat = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)

print("Accuracy :", resultat)

Accuracy : 0.956140350877193


# Phase 3 

In [10]:
def arene(X_train, X_test, y_train, y_test):
    modeles = {
        "Régression logistique": LogisticRegression(max_iter=10000),
        "Arbre de décision": DecisionTreeClassifier(random_state=42),
        "KNN": KNeighborsClassifier()
    }

    classement = []

    for nom, modele in modeles.items():
        accuracy = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
        classement.append((nom, accuracy))

    classement.sort(key=lambda resultat: resultat[1], reverse=True)

    for position, (nom, accuracy) in enumerate(classement, start=1):
        print(f"{position}. {nom} : {accuracy * 100:.2f} %")

    return classement

In [11]:
classement = arene(X_train, X_test, y_train, y_test)

1. Régression logistique : 95.61 %
2. KNN : 95.61 %
3. Arbre de décision : 94.74 %


# Phase 4 

In [12]:
def clustering_aveugle(X):
    modele = KMeans(n_clusters=2, random_state=42)
    labels_clusters = modele.fit_predict(X)
    return labels_clusters

In [13]:
labels_clusters = clustering_aveugle(X)

print("Répartition des clusters :", np.bincount(labels_clusters))

Répartition des clusters : [131 438]


In [14]:
score_direct = accuracy_score(y, labels_clusters)
score_inverse = accuracy_score(y, 1 - labels_clusters)
score_clustering = max(score_direct, score_inverse)

print(f"Correspondance avec les vraies classes : {score_clustering * 100:.2f} %")

Correspondance avec les vraies classes : 85.41 %


# Phase 5 : changer de terrain

In [15]:
X_wine, y_wine = load_wine(return_X_y=True)

In [16]:
explorer_dataset(X_wine, y_wine)

Forme des données : (178, 13)
Classes présentes : {np.int64(0), np.int64(1), np.int64(2)}
Classe 0 : 59 cas (33.15 %)
Classe 1 : 71 cas (39.89 %)
Classe 2 : 48 cas (26.97 %)


In [17]:
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.2, random_state=42, stratify=y_wine
)

In [18]:
classement_wine = arene(X_train_wine, X_test_wine, y_train_wine, y_test_wine)

1. Régression logistique : 94.44 %
2. Arbre de décision : 94.44 %
3. KNN : 80.56 %


## Observation

Les mêmes fonctions fonctionnent sans modification avec les 3 classes du dataset Wine. Le classement peut changer, car chaque algorithme réagit différemment selon les données.